# LLM 입력을 위한 최종 개선 프롬프트 생성 노트북 (Advanced v1)

이 노트북은 System/User 역할 분리, 동적 지시문, 페르소나 강화, 생각의 사슬(CoT) 등 모든 고급 기법을 적용하여 최종 프롬프트 파일(`advanced_prompts_for_llm.jsonl`)을 생성합니다.

In [1]:
import json
import pandas as pd

# --- 파일 경로 설정 ---
PRODUCT_FILE = 'product_info_preprocessed.jsonl'
PERSONA_FILE = 'persona_attributes_weighted.jsonl'
COMPETITOR_FILE_CLEANED = 'competitor_prices_cleaned_final.csv'
OUTPUT_FILE = 'advanced_prompts_for_llm.jsonl'

## 1. 모든 데이터 불러오기

In [2]:
# 제품 데이터
with open(PRODUCT_FILE, 'r', encoding='utf-8') as f:
    products = [json.loads(line) for line in f]
print(f"제품 {len(products)}개 불러오기 완료.")

# 페르소나 데이터
with open(PERSONA_FILE, 'r', encoding='utf-8') as f:
    personas = [json.loads(line) for line in f]
print(f"페르소나 {len(personas)}개 불러오기 완료.")

# 경쟁사 시장 데이터
try:
    df_competitor = pd.read_csv(COMPETITOR_FILE_CLEANED)
    market_context = {}
    for category, group in df_competitor.groupby('category'):
        min_price = group['price_per_100g'].min()
        max_price = group['price_per_100g'].max()
        market_context[category] = {
            "competitor_price_range_per_100g": f"{int(min_price):,}원 ~ {int(max_price):,}원"
        }
    print(f"경쟁사 시장 데이터 불러오기 완료. (카테고리: {list(market_context.keys())})")
except FileNotFoundError:
    print(f"오류: '{COMPETITOR_FILE_CLEANED}' 파일을 찾을 수 없습니다.")
    market_context = None

제품 15개 불러오기 완료.
페르소나 363개 불러오기 완료.
경쟁사 시장 데이터 불러오기 완료. (카테고리: ['RTD_액상커피', '그릭요거트', '참치액', '참치캔', '캔햄'])


## 2. 프롬프트 생성을 위한 헬퍼 함수 정의

In [3]:
FINAL_CATEGORY_MAPPING = {
    '참치 > 참치캔 > 라이트스탠다드참치': '참치캔', '참치 > 참치캔 > 가미참치': '참치캔',
    '조미소스 > 조미료 > 액상조미료': '참치액',
    '우유류 > 발효유 > 호상-중대용량': '그릭요거트',
    '축산 > 햄/소시지 > 캔햄': '캔햄', '수산 > 수산캔 > 번데기/골뱅이/꽁치': '캔햄',
    '우유류 > 커피 > 커피-CUP': 'RTD_액상커피'
}

def get_main_category(detailed_category):
    if detailed_category in FINAL_CATEGORY_MAPPING: return FINAL_CATEGORY_MAPPING[detailed_category]
    if '참치캔' in detailed_category: return '참치캔'
    if '참치액' in detailed_category or '액상조미료' in detailed_category: return '참치액'
    if '그릭' in detailed_category or '발효유' in detailed_category: return '그릭요거트'
    if '캔햄' in detailed_category: return '캔햄'
    if '커피' in detailed_category: return 'RTD_액상커피'
    return None

ATTRIBUTE_TRANSLATION = {
    "health_orientation_scaled": "건강 지향성", "price_sensitivity_scaled": "가격 민감도",
    "premium_orientation_scaled": "프리미엄 지향성", "variety_seeking_scaled": "다양성 추구",
    "cooking_convenience_scaled": "요리 편리성", "brand_loyalty_scaled": "브랜드 충성도",
    "hmr_preference_scaled": "HMR 선호도"
}

def get_most_important_attribute(persona_info):
    """페르소나의 구매 성향 속성 중 가중치가 가장 높은 속성의 이름을 한국어로 반환합니다."""
    attributes = persona_info.get('attributes', {})
    behavior_attributes = {k: v.get('weight', 0) for k, v in attributes.items() if '_scaled' in k}
    if not behavior_attributes: return "(분석 불가)"
    highest_weight_attribute_key = max(behavior_attributes, key=behavior_attributes.get)
    return ATTRIBUTE_TRANSLATION.get(highest_weight_attribute_key, highest_weight_attribute_key)

print("헬퍼 함수가 정의되었습니다.")

헬퍼 함수가 정의되었습니다.


## 3. 최종 개선 프롬프트 생성 및 저장

In [4]:
def create_advanced_prompt(product_info, persona_info, market_context):
    # --- System Prompt 생성 ---
    system_prompt = "당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 데이터를 종합적으로 분석하여, 가상 소비자의 구매 행동을 정밀하게 예측하고 그 결과를 JSON 객체로 생성하는 것입니다."

    # --- User Prompt 생성 ---
    product_str = json.dumps({k: product_info.get(k) for k in ['brand', 'product_name', 'category', 'features', 'targeted_consumer', 'price_text', 'advertise_info']}, ensure_ascii=False, indent=4)
    persona_str = json.dumps(persona_info, ensure_ascii=False, indent=4)
    main_category_key = get_main_category(product_info.get('category', ''))
    context_data = market_context.get(main_category_key, {})
    context_str = json.dumps(context_data, ensure_ascii=False, indent=4) if context_data else "{}"
    most_important_attr = get_most_important_attribute(persona_info)

    user_prompt = f"""# INSTRUCTION (Advanced)
먼저, 아래 '사고 과정(Chain-of-Thought)' 순서에 따라 당신의 분석을 단계별로 서술하세요.
1. **페르소나 핵심 분석**: 주어진 페르소나의 `meta` 정보와 가중치가 가장 높은 속성인 **'{most_important_attr}'** 를 바탕으로, 이 소비자의 가장 중요한 구매 동기가 무엇인지 정의합니다.
2. **제품-페르소나 연결**: 위 분석을 바탕으로, '제품 정보'의 특징들이 이 페르소나에게 얼마나 매력적일지 긍정적/부정적 요인을 평가합니다.
3. **시장 상황 적용**: '시장 경쟁 환경'의 가격 정보를 페르소나의 관점에서 어떻게 받아들일지 분석합니다.
4. **종합 결론**: 위의 모든 분석을 종합하여, 구매 확률과 월별 구매 빈도에 대한 최종 예측을 내립니다.

마지막으로, '---' 구분선 아래에 당신의 최종 예측을 # OUTPUT FORMAT에 맞는 JSON 형식으로만 생성하세요.

# INPUT DATA
## 1. 제품 정보
{product_str}

## 2. 페르소나 데이터
{persona_str}

## 3. 시장 경쟁 환경
{context_str}

---
# OUTPUT FORMAT
{{{{
  \"product_name\": \"{product_info.get('product_name', '')}\",
  \"persona_key\": {persona_info.get('persona_key')},
  \"purchase_behavior_prediction\": {{{{
    \"purchase_probability_pct\": \"<여기에 구매 확률(0-100)을 숫자로 예측>\",
    \"reason\": \"<여기에 페르소나 속성과 제품 특징, 시장 상황을 연결한 구매 결정 이유를 상세히 서술>\",
    \"monthly_purchase_frequency\": {{{{"<광고나 계절성이 없는 평범한 달의 월평균 예상 구매 개수를 숫자로 예측>"
    }}}}
  }}}}
}}}} """
    return system_prompt, user_prompt.strip()

if market_context is not None:
    print("프롬프트 생성을 시작합니다...")
    all_advanced_prompts = []
    for product in products:
        for persona in personas:
            system_prompt, user_prompt = create_advanced_prompt(product, persona, market_context)
            all_advanced_prompts.append({
                "product_name": product.get("product_name"),
                "persona_key": persona.get("persona_key"),
                "system_prompt": system_prompt,
                "user_prompt": user_prompt
            })

    print(f"\n생성된 프롬프트를 '{OUTPUT_FILE}' 파일로 저장합니다...")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for item in all_advanced_prompts:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("="*40)
    print("🎉 모든 작업이 성공적으로 완료되었습니다!")
    print(f"총 {len(all_advanced_prompts)}개의 프롬프트가 생성되어 '{OUTPUT_FILE}'에 저장되었습니다.")

    # --- 결과 미리보기 ---
    print("\n--- 생성된 첫 번째 프롬프트 예시 ---")
    if all_advanced_prompts:
        example = all_advanced_prompts[0]
        print("\n[SYSTEM PROMPT]")
        print(example['system_prompt'])
        print("\n[USER PROMPT]")
        print(example['user_prompt'])
else:
    print("시장 데이터가 로드되지 않아 프롬프트 생성을 건너뜁니다.")

프롬프트 생성을 시작합니다...

생성된 프롬프트를 'advanced_prompts_for_llm.jsonl' 파일로 저장합니다...
🎉 모든 작업이 성공적으로 완료되었습니다!
총 5445개의 프롬프트가 생성되어 'advanced_prompts_for_llm.jsonl'에 저장되었습니다.

--- 생성된 첫 번째 프롬프트 예시 ---

[SYSTEM PROMPT]
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 데이터를 종합적으로 분석하여, 가상 소비자의 구매 행동을 정밀하게 예측하고 그 결과를 JSON 객체로 생성하는 것입니다.

[USER PROMPT]
# INSTRUCTION (Advanced)
먼저, 아래 '사고 과정(Chain-of-Thought)' 순서에 따라 당신의 분석을 단계별로 서술하세요.
1. **페르소나 핵심 분석**: 주어진 페르소나의 `meta` 정보와 가중치가 가장 높은 속성인 **'HMR 선호도'** 를 바탕으로, 이 소비자의 가장 중요한 구매 동기가 무엇인지 정의합니다.
2. **제품-페르소나 연결**: 위 분석을 바탕으로, '제품 정보'의 특징들이 이 페르소나에게 얼마나 매력적일지 긍정적/부정적 요인을 평가합니다.
3. **시장 상황 적용**: '시장 경쟁 환경'의 가격 정보를 페르소나의 관점에서 어떻게 받아들일지 분석합니다.
4. **종합 결론**: 위의 모든 분석을 종합하여, 구매 확률과 월별 구매 빈도에 대한 최종 예측을 내립니다.

마지막으로, '---' 구분선 아래에 당신의 최종 예측을 # OUTPUT FORMAT에 맞는 JSON 형식으로만 생성하세요.

# INPUT DATA
## 1. 제품 정보
{
    "brand": "동원 F&B",
    "product_name": "덴마크 하이그릭요거트 400g",
    "category": "우유류 > 발효유 > 호상-중대용량",
    "features": [
        "건강식품",
        "고단백",
        "고소한